In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
import spacy
from datasets import load_dataset
import json

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Load the dataset
dataset = load_dataset("ankurani/Factify5wqa", data_files="5WQA_all_claims_with_evidence.json.zip")
claims = dataset['train']  # Assuming the data is in the 'train' split

# Function to extract 5W aspects using spaCy
def extract_5w_spacy(claim):
    doc = nlp(claim)
    aspects = {"who": None, "what": None, "when": None, "where": None, "why": None}

    # Extract "Who" from named entities (e.g., ORGANIZATION, PERSON)
    aspects["who"] = [ent.text for ent in doc.ents if ent.label_ in ["ORG", "PERSON"]]

    # Extract "When" from named entities (e.g., DATE, TIME)
    aspects["when"] = [ent.text for ent in doc.ents if ent.label_ in ["DATE", "TIME"]]

    # Extract "Where" from named entities (e.g., GPE, LOC)
    aspects["where"] = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]

    # Extract "What" from noun chunks (simplified approach)
    aspects["what"] = [chunk.text for chunk in doc.noun_chunks if "what" in chunk.text.lower()]

    # Extract "Why" using causal phrases (rule-based, simplified)
    for token in doc:
        if token.dep_ == "mark" and token.text.lower() in ["because", "due to", "as"]:
            aspects["why"] = token.head.text
            break

    return aspects

# Process the dataset
processed_claims = []

for entry in claims:
    claim = entry["claim"]
    evidence = entry["evidence"]
    print(f"Processing: {claim[:50]}...")  # Log progress
    aspects = extract_5w_spacy(claim)

    processed_claims.append({
        "claim": claim,
        "evidence": evidence,
        "5W_aspects": aspects
    })

# Save results to a JSON file
output_filepath = "processed_claims_spacy_5w.json"
with open(output_filepath, 'w', encoding='utf-8') as f:
    json.dump(processed_claims, f, ensure_ascii=False, indent=4)

print(f"Processed data saved to {output_filepath}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.40k [00:00<?, ?B/s]

5WQA_all_claims_with_evidence.json.zip:   0%|          | 0.00/266M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Streaming output truncated to the last 5000 lines.
Processing: Google was established while Page and Brin were Ph...
Processing: Moscow possesses political institutions....
Processing: The Golden State Warriors play home games in Oakla...
Processing: Matt Damon portrayed Scott Thorson in a movie....
Processing: War Dogs was co-directed by Bradley Cooper....
Processing: Taraji P. Henson starred in a 2012 American romant...
Processing: Brown bears are not carnivores....
Processing: Justinian I died in 565....
Processing: James Blake (musician) released his third album in...
Processing: The Weeknd published his third novel in 2016....
Processing: Thomas DeSimone was born on May 24....
Processing: Ryan Phillippe has a child....
Processing: Francois de Belleforest translated the works of De...
Processing: Split (2016 American film) stars Batman....
Processing: Julia Louis-Dreyfus played Christine Campbell in S...
Processing: Rear Window (1998 film) was one of Christopher Ree...
Processing: 

In [ ]:
# Load the processed JSON file
with open("processed_claims_spacy_5w.json", 'r', encoding='utf-8') as f:
    processed_data = json.load(f)

# Display a sample of the processed claims
for entry in processed_data[:5]:  # Display the first 5 entries
    print(f"Claim: {entry['claim']}")
    print(f"Evidence: {entry['evidence']}")
    print(f"5W Aspects: {entry['5W_aspects']}")
    print("-" * 50)


Claim: China’s famed wandering elephants are on the move again, heading southwest while a male who broke from the herd is still keeping his distance. https://t.co/o5j7PDDveJ
Evidence: By Julia Hollingsworth and Zixu Wang, CNNUpdated 1:03 AM ET, Fri June 11, 2021  (CNN)At least a dozen buzzing drones monitor them around the clock.  Wherever they go, they're escorted by police.  And when they eat or sleep, they're watched by millions online.  CNN's Jessie Yeung contributed to this report.  
5W Aspects: {'who': [], 'what': [], 'when': [], 'where': ['China'], 'why': None}
--------------------------------------------------
Claim: Russia: Prime Minister Narendra Modi and Russian President Vladimir Putin at Zvezda ship-building complex, Vladivostok. https://t.co/j128uqqNw0
Evidence: Russian President Vladimir Putin on Wednesday hosted Indian Prime Minister Narendra Modi for talks on boosting investment and trade, with a special emphasis on energy and arms deals. The two met on the sidelines o

In [ ]:
import pandas as pd

# Create a DataFrame from the processed data
df = pd.DataFrame(processed_data)

# Analyze frequency of non-null aspects
aspect_counts = df['5W_aspects'].apply(pd.Series).notnull().sum()
print("Frequency of Extracted 5W Aspects:")
print(aspect_counts)


Frequency of Extracted 5W Aspects:
who      391041
what     391041
when     391041
where    391041
why        2637
dtype: int64


# ***Unlearning***

In [3]:
!pip install transformers datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [1]:
import json

# Load your processed dataset
with open("processed_claims_spacy_5w.json", 'r', encoding='utf-8') as f:
    processed_data = json.load(f)

print(f"Loaded dataset with {len(processed_data)} entries.")


Loaded dataset with 391041 entries.


In [2]:
# Convert "Who" labels to strings for classification
who_data = {
    "claim": [entry["claim"] for entry in processed_data if entry["5W_aspects"].get("who")],
    "label": ["; ".join(entry["5W_aspects"]["who"]) if isinstance(entry["5W_aspects"]["who"], list) else entry["5W_aspects"]["who"]
              for entry in processed_data if entry["5W_aspects"].get("who")]
}

# Verify the processed labels
print("Sample Labels:", who_data["label"][:5])


Sample Labels: ['Narendra Modi; Vladimir Putin; Zvezda; Vladivostok', "Mahatama Gandhi's; CCTV", 'Dhanbad; COVID19', "Neil Gorsuch; the Supreme Court's; LGBTQ", 'Ravi Shankar Prasad; AIIMS Patna']


In [3]:
from datasets import Dataset

# Convert to Hugging Face Dataset format
who_dataset = Dataset.from_dict(who_data)

# Split into train/test sets
who_dataset = who_dataset.train_test_split(test_size=0.2)
train_who_dataset = who_dataset["train"]
test_who_dataset = who_dataset["test"]

print(f"Training samples: {len(train_who_dataset)}, Testing samples: {len(test_who_dataset)}")


Training samples: 221880, Testing samples: 55471


In [4]:
from transformers import AutoTokenizer

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples["claim"], truncation=True, padding="max_length", max_length=128)

# Tokenize the datasets
tokenized_train_who = train_who_dataset.map(tokenize_function, batched=True)
tokenized_test_who = test_who_dataset.map(tokenize_function, batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/221880 [00:00<?, ? examples/s]

Map:   0%|          | 0/55471 [00:00<?, ? examples/s]

In [5]:
# Number of unique labels
num_labels = len(set(who_data["label"]))
print(f"Number of unique labels for 'Who': {num_labels}")


Number of unique labels for 'Who': 96269


In [6]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu


In [7]:
!pip install --upgrade transformers

In [8]:
from transformers import AutoModelForSequenceClassification

# Number of unique labels (distinct "Who" entries)
num_labels = len(set(who_data["label"]))

# Load pre-trained DistilBERT model with classification head
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

print(f"Model loaded with {num_labels} output labels.")


/usr/local/lib/python3.11/dist-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.11/dist-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with 96269 output labels.


In [11]:
# Create a mapping from unique labels to integers
unique_labels = list(set(who_data["label"]))
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}

# Apply the mapping to convert labels to integers
who_data["label"] = [label_to_id[label] for label in who_data["label"]]

# Verify the conversion
print(f"Label Mapping: {label_to_id}")
print(f"Sample Labels: {who_data['label'][:5]}")


Label Mapping: {'Jessica Lange; Golden Globe; Rent': 0, "John D. Odegard School of Aerospace Sciences'; The Texas Air & Space Museum": 1, 'British Airways; BA; EasyJet': 2, 'Erik Spoelstra': 3, 'Cotonou; Benin': 4, 'Susan Sarandon; the Oscar Award': 5, 'Baghel; Jawahar Lal Nehru Memorial': 6, 'Steven Moffat': 7, 'the Crown Jewels': 8, 'The Celtic F.C.': 9, 'Promethus; Academy Award; BAFTA': 10, 'Helena Bonham Carter; Sweeney Todd': 11, 'Gracepoint': 12, 'La Rioja; COVID-19': 13, 'Christian BlackMartin': 14, "Joe Biden's; State; Biden": 15, 'Local H': 16, 'Jerry Goldsmith; British Academy Film Award': 17, 'William Percival; the Seattle Seahawks': 18, 'n Dilwale Dulhania Le Jayenge , Shah Rukh Khan; Malhotra': 19, "Teletoon Retro; OWN; Oprah Winfrey Network; Corus Entertainment 's": 20, 'Cruel Intentions': 21, 'Kellerman Log Cabin': 22, 'Flash Thompson; Peter Parker; Spider': 23, 'Lisa; Sharon': 24, 'Voter; Himachal; Madhya Pradesh-7.16%': 25, 'the House of Blues': 26, 'Douglas Aircraft 

In [12]:
from datasets import Dataset

# Convert the data to a Hugging Face Dataset
who_dataset = Dataset.from_dict(who_data)

# Split the dataset into train and test sets
who_dataset = who_dataset.train_test_split(test_size=0.2)
train_who_dataset = who_dataset["train"]
test_who_dataset = who_dataset["test"]

print(f"Training samples: {len(train_who_dataset)}, Testing samples: {len(test_who_dataset)}")


Training samples: 221880, Testing samples: 55471


In [13]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["claim"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized_inputs["labels"] = examples["label"]  # Ensure labels are included
    return tokenized_inputs

# Tokenize the datasets
tokenized_train_who = train_who_dataset.map(tokenize_function, batched=True)
tokenized_test_who = test_who_dataset.map(tokenize_function, batched=True)

# Verify the format
print(tokenized_train_who[0])


Map:   0%|          | 0/221880 [00:00<?, ? examples/s]

Map:   0%|          | 0/55471 [00:00<?, ? examples/s]

{'claim': 'Breshad Perriman played high school football for the Rams .', 'label': 85820, 'input_ids': [101, 7987, 9953, 4215, 2566, 20026, 2319, 2209, 2152, 2082, 2374, 2005, 1996, 13456, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': 85820}


In [15]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load the model with the correct number of labels
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(label_to_id)
)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./who_results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./who_logs"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_who,
    eval_dataset=tokenized_test_who,
    tokenizer=tokenizer
)

# Train the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("./fine_tuned_who")
print("Fine-tuned model for 'Who' saved!")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-15-b766004344d8>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING Serializing object of type dict that is 5242960 bytes
wandb: WARNING Serializing object of type dict that is 3844864 bytes


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 